# 1. Data Pipeline Component
Bu bölümde verinin yüklenmesi, NLP standartlarına göre ön işlemeden (preprocessing) geçirilmesi, TF-IDF ile vektörize edilmesi ve Train/Test olarak ayrılması işlemleri gerçekleştirilmiştir.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# NLTK Paketleri
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

# ---------------------------------------------------------
# 1. DATA PIPELINE COMPONENT
# ---------------------------------------------------------
def load_data():
    url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
    df = pd.read_csv(url, sep='\t', header=None, names=['Label', 'Text'])
    df = df[['Text', 'Label']]
    df['Label'] = df['Label'].map({'ham': 'Not Spam', 'spam': 'Spam'})
    return df

df = load_data()

# ---------------------------------------------------------
# 2. TEXT PREPROCESSING COMPONENT
# ---------------------------------------------------------
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return ' '.join(tokens)

df['Clean_Text'] = df['Text'].apply(clean_text)

# ---------------------------------------------------------
# 3. FEATURE ENGINEERING COMPONENT
# ---------------------------------------------------------
tfidf_model = TfidfVectorizer(max_features=3000)
X_features = tfidf_model.fit_transform(df['Clean_Text'])

# Train/Test Split (Modelleme için hazırlık)
y = df['Label'].map({'Spam': 1, 'Not Spam': 0})
X_train, X_test, y_train, y_test = train_test_split(X_features, y, test_size=0.2, random_state=42, stratify=y)

print("Veri yükleme, temizleme ve özellik çıkarımı (TF-IDF) başarıyla tamamlandı!")
print(f"Eğitim Seti: {X_train.shape[0]} satır | Test Seti: {X_test.shape[0]} satır")

# 4. Model Component & 5. Evaluation Component
Zorunlu tutulan Naive Bayes ve destekleyici model olarak Support Vector Machine (SVM) algoritmalarının eğitilmesi, performans metriklerinin hesaplanması ve Confusion Matrix görselleştirmeleri.

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import time

models = {
    "Naive Bayes": MultinomialNB(),
    "Support Vector Machine (SVM)": SVC(kernel='linear', random_state=42)
}

results = {}
trained_models = {}

for model_name, model in models.items():
    start_time = time.time()
    model.fit(X_train, y_train)
    end_time = time.time()
    
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    results[model_name] = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1}
    trained_models[model_name] = model
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Not Spam', 'Spam'], yticklabels=['Not Spam', 'Spam'])
    plt.title(f'{model_name} - Confusion Matrix')
    plt.ylabel('Gerçek Değerler')
    plt.xlabel('Tahmin Edilen Değerler')
    plt.show()

# Karşılaştırma Tablosu
print("\n--- MODEL KARŞILAŞTIRMA TABLOSU ---")
results_df = pd.DataFrame(results).T
print(results_df.round(4).to_string())

# 6. Inference (Prediction) Component
Eğitilen modelin, dışarıdan gelen yeni ve görülmemiş e-postalar üzerinde Spam / Not Spam tahmini yapmasını sağlayan test modülü.

In [ ]:
def predict_spam(email_text, model, vectorizer):
    cleaned_text = clean_text(email_text)
    vectorized_text = vectorizer.transform([cleaned_text])
    prediction = model.predict(vectorized_text)
    return "Spam" if prediction[0] == 1 else "Not Spam"

# Şirketin istediği 3 uç test senaryosu
test_emails = [
    "Hi team, let's schedule a meeting for tomorrow at 10 AM to discuss the new project.",
    "CONGRATULATIONS!! You have been selected to receive a FREE $1000 Walmart Gift Card. Click here to claim your prize now!",
    "Hey bro, are we still going to the cinema tonight? Call me when you see this."
]

selected_model = trained_models["Support Vector Machine (SVM)"]

for i, email in enumerate(test_emails, 1):
    result = predict_spam(email, selected_model, tfidf_model)
    print(f"Test {i}:")
    print(f"E-Posta: '{email}'")
    print(f"Tahmin : {result}\n")